In [ ]:
try:
    from google.colab import drive
    drive.mount('/gdrive')
    dataset_root = '/gdrive/MyDrive/datasets'
    !pip install torchinfo tqdm
    colab = True
except Exception as e:
    print(e)
    print('Assuming we\'re not on colab.')
    dataset_root = './datasets'
    colab = False

print('Will store datasets in', dataset_root)

import os

if os.name == 'nt':
    print("Disabling multiprocessing because we're running on windows.")
    cpu_num = 0
elif colab:
    cpu_num = 2
else:
    cpu_num = os.cpu_count() // 2
    print('Dataloaders will use {} CPUs'.format(cpu_num))

In [ ]:
import random

import torch
import torch.utils.data as tud
import torch.nn as nn
import torch.nn.functional as F

import torchvision.transforms as tvt
import torchvision.transforms.functional as tvf
import torchvision.datasets as tds
import torchvision.utils as tu

from tqdm import tqdm
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# 20 classes, as well as none.
classes = {
    0:'None',
    1:'Aeroplane',
    2:'Bicycle',
    3:'Bird',
    4:'Boat',
    5:'Bottle',
    6:'Bus',
    7:'Car',
    8:'Cat',
    9:'Chair',
    10:'Cow',
    11:'Diningtable',
    12:'Dog',
    13:'Horse',
    14:'Motorbike',
    15:'Person',
    16:'Pottedplant',
    17:'Sheep',
    18:'Sofa',
    19:'Train',
    20:'Tvmonitor',
}
# num_classes = len(classes)
num_classes = 2

In [ ]:
class OneHotMap(nn.Module):
    def __init__(self, num_classes, trunc_mode='clip'):
        super().__init__()
        self.num_classes = num_classes
        self.trunc_mode = trunc_mode

    def forward(self, x):
        if self.trunc_mode == 'clip':
            # Any class higher than num_classes is set to the highest class number.
            x = torch.clamp_max(x, self.num_classes - 1)
        elif self.trunc_mode == 'background':
            # Any class higher than num_classes is set to background
            x[x > (self.num_classes -1)] = 0
        else:
            raise Exception("Invalid truncation mode")
        
        x = F.one_hot(x.long(), self.num_classes)
        return x.squeeze(0).permute(2, 0, 1)

In [ ]:
train_tfs = tvt.Compose([
    tvt.PILToTensor(),
    tvt.Resize((256,256), antialias=True)
])

eval_tfs = tvt.Compose([
    tvt.PILToTensor(),
    tvt.Resize((256,256), antialias=True)
])

label_tfs = tvt.Compose([
    tvt.PILToTensor(),
    tvt.Resize((256,256), antialias=False),
    OneHotMap(num_classes, trunc_mode='clip')
])

voc_train = tds.VOCSegmentation(
    root=dataset_root,
    download=True,
    year='2012',
    image_set='train',
    transform=train_tfs,
    target_transform=label_tfs
)

voc_val = tds.VOCSegmentation(
    root=dataset_root,
    download=True,
    year='2012',
    image_set='val',
    transform=eval_tfs,
    target_transform=label_tfs
)


In [ ]:
def random_grid(imgs, sz: int):
    grid = tu.make_grid(imgs)
    return grid.permute(1, 2, 0)

num=64
augmented = torch.stack([x[0] for x in random.choices(voc_train, k=num)])
tmps = random_grid(augmented, num)
plt.imshow(tmps.cpu())

In [ ]:
batchsize = 16

train_loader = tud.DataLoader(
    voc_train,
    batch_size=batchsize,
    num_workers=cpu_num,
    shuffle=True)
val_loader = tud.DataLoader(
    voc_val,
    batch_size=batchsize,
    num_workers=cpu_num,
    shuffle=True)

In [ ]:
class SimpleDownConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv_1 = nn.Conv2d(cin, cout, kernel_size=3, stride=1, padding=1)
        self.bn_1 = nn.BatchNorm2d(cout)
        self.conv_2 = nn.Conv2d(cout, cout, kernel_size=3, stride=1, padding=1)
        self.bn_2 = nn.BatchNorm2d(cout)

    def forward(self, x):
        x = self.conv_1(x)
        x = F.relu(x)
        x = self.bn_1(x)
        
        x = self.conv_2(x)
        x = F.relu(x)
        x = self.bn_2(x)

        x = F.max_pool2d(x, kernel_size=2) 
        return x

class SimpleUpConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.upconv_1 = nn.ConvTranspose2d(cin, cout, kernel_size=2, stride=2)
        self.bn_1 = nn.BatchNorm2d(cout)
        self.conv_2 = nn.Conv2d(cout, cout, kernel_size=3, stride=1, padding=1)
        self.bn_2 = nn.BatchNorm2d(cout)
        self.conv_3 = nn.Conv2d(cout, cout, kernel_size=3, stride=1, padding=1)
        self.bn_3 = nn.BatchNorm2d(cout)

    def forward(self, x):
        x = self.upconv_1(x)
        x = self.bn_1(x)
        
        x = self.conv_2(x)
        x = F.relu(x)
        x = self.bn_2(x)

        x = self.conv_3(x)
        x = F.relu(x)
        x = self.bn_3(x)
        
        return x
    
class SimpleSeg(nn.Module):
    def __init__(self, cin, cout=num_classes, filts=32):
        super().__init__()
        num_filters=filts
        self.l1 = SimpleDownConv(cin, num_filters)
        self.l2 = SimpleDownConv(num_filters, num_filters*2)
        self.l3 = SimpleDownConv(num_filters*2, num_filters*4)
    
        self.l4 = SimpleUpConv(num_filters*4, num_filters*2)
        self.l5 = SimpleUpConv(num_filters*2, num_filters)
        self.l6 = SimpleUpConv(num_filters, num_filters)
        self.l7 = nn.Conv2d(num_filters, num_classes, kernel_size=1, stride=1)

    def forward(self, x):
        # Down
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)

        # Up
        x = self.l4(x)
        x = self.l5(x)
        x = self.l6(x)

        # Output
        x = self.l7(x)
        return x


In [ ]:
from torchinfo import summary
testmodel = SimpleSeg(3, num_classes)
lossfn = nn.CrossEntropyLoss()
print(summary(testmodel, (1, 3, 512, 512)))

In [ ]:
%%time

model = SimpleSeg(3, num_classes, filts=8).to(device).train()

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
epochs = 50
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

lossfn = nn.CrossEntropyLoss()

loss_plot = []
try:
    for epoch in range(epochs):
        model.train()
        for i, (images, target) in enumerate(tqdm(train_loader)):
            optimizer.zero_grad()
            images = images.float().to(device)
            targets = target.float().to(device)
    
            outs = model(images)
            loss = lossfn(outs, targets)
            loss.backward()
            optimizer.step()

        if epoch % 5 == 0:
            losses = []
            model.eval()
            correct = 0
            total = len(voc_val)
            print("Running val...")
            with torch.no_grad():
                for i, (images, target) in enumerate(tqdm(val_loader)):
                    images = images.float().to(device)
                    targets = target.float().to(device)
                    outs = model(images)
        
                    loss = lossfn(outs, targets)
                    losses.append(loss.cpu().item())
                        
            epoch_loss = torch.Tensor(losses).mean().item()
            print("Epoch {}: {} ".format(epoch, epoch_loss))
            print("Current LR is {}".format(scheduler.get_last_lr()))
            loss_plot.append(epoch_loss)
        scheduler.step()
        
except KeyboardInterrupt:
    plt.plot(loss_plot)
plt.plot(loss_plot)

In [ ]:
from ipywidgets import interact

onehot_transform = OneHotMap(num_classes, trunc_mode='clip')

@interact(index=(0, len(voc_val)), thresh=(0, 1, 0.01))
def draw_preds(index=0, thresh=0.5):
    data, label = voc_val[index]
    input = data.float().cuda().unsqueeze(0)
    with torch.no_grad():
        model.eval()
        pred = model(input)
        pred = F.softmax(pred, dim=1)
        pred = torch.threshold(pred, thresh, 0)
        pred[pred > 0.001] = 1
    
    fig, (ax1, ax2) = plt.subplots(1,2)
    ax1.imshow(data.permute(1,2,0))
    ax1.imshow(label.squeeze()[1], alpha=0.4)
    
    ax2.imshow(data.permute(1,2,0))
    ax2.imshow(pred[0, 1, ...].cpu(), alpha=0.4)